In [1]:
#pip install PyMuPDF

In [2]:
import json
import os
import pprint
import sys
from base64 import b64encode
from typing import Dict, List, Union

from openai import AzureOpenAI
from openai.types.chat import ChatCompletion
from pymupdf import Document, Matrix, Pixmap
from pymupdf import open as pdf_open
"""
from communs.exceptions.ExperimentException import ExperimentException
from communs.ressources.settings import Settings
from document_classification.doc_classification_experiment import (
    DocClassificationExperiment,
)
from document_classification.enum.content_type_labels import ContentTypeLabels
from document_classification.enum.document_sub_type_labels import DocumentSubTypeLabels
from document_classification.enum.document_type_labels import DocumentTypeLabels
from document_classification.enum.result_keys import ResultKeys
"""

from dotenv import load_dotenv
load_dotenv("../ez_env.env")


True

In [3]:
endpoint = os.environ.get("OPENAI_API_BASE")
api_key = os.environ.get("OPENAI_API_KEY")
deployment = os.environ.get("DEPLOYMENT_NAME")
openai_api_version = os.environ.get("OPENAI_API_VERSION")


In [4]:
openai_api_version

'2024-06-01'

In [5]:


_debug_pdf=False

def processing_file( file_experiment: str,path:str) :
    results: Dict[str, str] = dict()

    
    current_filename: str = file_experiment
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=openai_api_version,
    )
    with open(file_experiment, "rb") as f:
        pdf_document: Document = pdf_open(stream=f.read(), filetype="pdf")  # open document
    pdf_pages_pixmap: List[Pixmap] = _extract_pixmap_pages_from_pdf(document_filename=file_experiment, pdf_document=pdf_document)
    chat_completion_message: List[Dict] = _create_chat_completion_message(pdf_pages_pixmap)
    
    for i in range(40):
        print(f"Step {i+1}- filname : {current_filename}-pages : {len(pdf_pages_pixmap)}")          
        chat_completion: ChatCompletion = client.chat.completions.create(
            model=deployment, messages=chat_completion_message, temperature=0, seed=42, response_format={"type": "json_object"}
        )

        chat_completion_content: Dict = json.loads(chat_completion.choices[0].message.content)
        print(chat_completion_content)

    return chat_completion_content

def _create_chat_completion_message( pdf_pages_pixmap: List[Pixmap]) -> List[Dict]:
    user_message: Dict = _get_default_user_message()
    for pdf_page_pixmap in pdf_pages_pixmap:
        print(pdf_page_pixmap)
        pixmap_page_encoded: str = _encode_pixmap_to_base64(pdf_page_pixmap)
        image_message: Dict = _get_default_image_message()
        image_message["image_url"]["url"] = f"data:image/png;base64,{pixmap_page_encoded}"
        user_message["content"].append(image_message)
    return [_get_system_message(), user_message]

def _encode_pixmap_to_base64( pixmap: Pixmap) -> str:
    return b64encode(pixmap.tobytes(jpg_quality=100)).decode("utf-8")

def _extract_pixmap_pages_from_pdf( document_filename: str, pdf_document: Document) -> List[Pixmap]:
    pdf_pages_pixmap: List[Pixmap] = list()
    for pdf_page in pdf_document:
        zoom_x = 4.0  # horizontal zoom
        zoom_y = 4.0  # vertical zoom
        mat = Matrix(zoom_x, zoom_y)
        pixmap: Pixmap = pdf_page.get_pixmap(matrix=mat)
        pdf_pages_pixmap.append(pixmap)
        if _debug_pdf:
            document_path = document_filename.split(".")[0]
            pixmap.save(f"{document_path}-page-{pdf_page.number}.png", jpg_quality=100)
    return pdf_pages_pixmap

def _get_default_user_message() -> Dict:
    return {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": """
                Can you tell me which class this document belongs to ?
            """,
            },
        ],
    }

def _get_default_image_message() -> Dict:
    return {
        "type": "image_url",
        "image_url": {"url": "", "detail": "high"},
    }

def _get_system_message() -> Dict:
    return {
        "role": "system",
        "content": """
            You are an AI assistant designed to classify documents.
            The user will submit this document to you. Every image submitted will be a page of the document.
            You have to consider every page to classify the document correctly.

            First, you must classify this document into 3 different classes relating to the "document type".
            The possible values for "document type" are : a "Gift Letter", a "Property Listing" or a "Seller's disclosure".
            Note that a property listing will describe all the known properties about a house, in a generic manner.
            A "seller's disclosure" is a self-report document that must be completed by the seller of a property.

            Second, if it is a "Seller's Disclosure", you must also classify it into 2 other classes relating to the "company name".
            The possible values for the "company name" are : "duProprio" or "OACIQ".

            Finally, you must classify this document into 2 other classes relating to the "content type".
            You have to tell if the document was "handwritten" or "typed".
            If a document entry is censored or emtpy, you should ignore it.
            If the document presents both classifications, return both classes.
            You must also explain why you chose this calligraphy classification.

            Present the extracted information in a structured JSON format, like so : 
            {
                "document_type": "Attestation relative à un don",
                "company_name": "OACIQ",
                "content_type": ["handwritten"],
            }
        """,
    }

"""
if __name__ == "__main__":
    args = sys.argv[1:]
    experiment = Experiment05()
    experiment.run_main(args)
"""    

'\nif __name__ == "__main__":\n    args = sys.argv[1:]\n    experiment = Experiment05()\n    experiment.run_main(args)\n'

In [9]:
processing_file(file_experiment='../in/WSP prop. HQP_Analyse_de_risque_2021-Ponts_roulants_signé.pdf',path="../out")

Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Pixmap(DeviceRGB, (0, 0, 2448, 3168), 0)
Step 1- filname : ../in/WSP prop. HQP_Analyse_de_risque_2021-Ponts_roulants_signé.pdf-pages : 17


PermissionDeniedError: Error code: 403 - {'error': {'code': '403', 'message': 'Access denied due to Virtual Network/Firewall rules.'}}